# 03 Búsqueda vectorial con TF-IDF

**Nombre:** Joshua Daniel Menendez Farias

**Fecha:** 11/11/2026

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [ ]:
import tarfile
import os
import re
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def preprocesar(texto):
    return re.findall(r'\b[a-z]+\b', texto.lower())

def extraer_titulo_gutenberg(texto_raw, doc_id):
    """
    Busca la cabecera 'Title:' típica de Project Gutenberg o toma la primera línea no vacía.
    """
    match = re.search(r'^Title:\s*(.+)$', texto_raw, re.MULTILINE | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    for line in texto_raw.splitlines():
        limpia = line.strip()
        if limpia and not limpia.startswith('***'):
            return limpia[:80]
    return doc_id

def cargar_corpus_grande(ruta_tar, max_docs=300):
    """
    Reutiliza la función de extracción de la práctica 02 (acotada a 300 documentos
    para evitar saturar la memoria RAM), conservando textos originales, títulos y tokens.
    Evita contar distintas versiones del mismo libro filtrando por ID numérico de Gutenberg.
    """
    corpus_tokens = {}
    textos_originales = {}
    titulos = {}
    libros_vistos = set()
    documentos_validos = 0
    
    if not os.path.exists(ruta_tar):
        print(f"Error: El archivo '{ruta_tar}' no se encuentra en el directorio actual.")
        return corpus_tokens, textos_originales, titulos

    with tarfile.open(ruta_tar, 'r') as tar:
        for member in tar:
            if documentos_validos >= max_docs:
                break
            
            if member.isfile() and member.name.endswith('.txt'):
                # Evitar versiones duplicadas del mismo libro (ej. 12345-0.txt, 12345-8.txt)
                nombre_base = os.path.basename(member.name)
                id_libro = nombre_base.split('-')[0].split('.')[0]
                if id_libro in libros_vistos:
                    continue
                
                f = tar.extractfile(member)
                if f:
                    try:
                        texto = f.read().decode('utf-8', errors='ignore')
                        tokens = preprocesar(texto)
                        if tokens:
                            doc_id = member.name
                            corpus_tokens[doc_id] = tokens
                            textos_originales[doc_id] = texto
                            titulos[doc_id] = extraer_titulo_gutenberg(texto, doc_id)
                            libros_vistos.add(id_libro)
                            documentos_validos += 1
                    except Exception:
                        continue
                        
    return corpus_tokens, textos_originales, titulos

# Ejecución sobre los 300 documentos del archivo txt-files.tar
ruta_tar = 'txt-files.tar'
corpus_tokens, textos_originales, titulos = cargar_corpus_grande(ruta_tar, max_docs=300)
docs_ids = list(corpus_tokens.keys())

filas_tabla = []
vocabulario_set = set()
longitudes = []

for doc_id in docs_ids:
    tokens = corpus_tokens[doc_id]
    terminos_unicos = set(tokens)
    vocabulario_set.update(terminos_unicos)
    longitudes.append(len(tokens))
    
    filas_tabla.append({
        'Identificador de documento': doc_id,
        'Título': titulos[doc_id],
        'Número de términos únicos': len(terminos_unicos)
    })

df_corpus = pd.DataFrame(filas_tabla)
display(df_corpus)

if longitudes:
    print(f"\n--- Estadísticas del Corpus ({len(docs_ids)} documentos) ---")
    print(f"Número de documentos: {len(docs_ids)}")
    print(f"Tamaño del vocabulario: {len(vocabulario_set)} términos únicos")
    print(f"Longitud mínima de los documentos: {np.min(longitudes)} tokens")
    print(f"Longitud media de los documentos: {np.mean(longitudes):.2f} tokens")
    print(f"Longitud máxima de los documentos: {np.max(longitudes)} tokens")

,Identificador de documento,Título,Número de términos únicos
0,cache/epub/10000/pg10000.txt,The Magna Carta,1788
1,cache/epub/10001/pg10001.txt,Apocolocyntosis,2058
2,cache/epub/10002/pg10002.txt,The House on the Borderland,5268
3,cache/epub/10003/pg10003.txt,"My First Years as a Frenchwoman, 1876-1879",6053
4,cache/epub/10004/pg10004.txt,The Warriors,6783
...,...,...,...
295,cache/epub/10388/pg10388.txt,"A Collection of Old English Plays, Volume 1",10003
296,cache/epub/10389/pg10389.txt,"Northern Trails, Book I.",4210
297,cache/epub/1038/pg1038.txt,Style,5646
298,cache/epub/10390/pg10390.txt,The Edinburgh Lectures on Mental Science,3504



--- Estadísticas del Corpus (300 documentos) ---
Número de documentos: 300
Tamaño del vocabulario: 277088 términos únicos
Longitud mínima de los documentos: 5262 tokens
Longitud media de los documentos: 75287.35 tokens
Longitud máxima de los documentos: 985957 tokens


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [2]:
def calcular_tf(corpus_tokens, vocabulario, docs_ids):
    """
    1. Calcula la matriz TF (N_docs x N_vocab) con conteo directo f(t, d).
    """
    term_to_idx = {termino: i for i, termino in enumerate(vocabulario)}
    matriz_tf = np.zeros((len(docs_ids), len(vocabulario)), dtype=np.float64)
    
    for d_idx, doc_id in enumerate(docs_ids):
        conteos = Counter(corpus_tokens[doc_id])
        for termino, frec in conteos.items():
            if termino in term_to_idx:
                matriz_tf[d_idx, term_to_idx[termino]] = frec
    return matriz_tf

def calcular_df(matriz_tf):
    """
    2. Calcula el vector DF contando en cuántos documentos aparece cada término al menos una vez.
    """
    return np.sum(matriz_tf > 0, axis=0).astype(np.float64)

def calcular_idf(df_vector, N):
    """
    3. Calcula IDF(t) = ln(N / DF(t)) + 1 (fórmula de scikit-learn con smooth_idf=False).
    """
    return np.log(N / df_vector) + 1.0

def calcular_tfidf(matriz_tf, idf_vector):
    """
    4. Calcula la matriz de pesos w_{t,d} = TF(t,d) * IDF(t).
    """
    return matriz_tf * idf_vector

print("Funciones explícitas para TF, DF, IDF y TF-IDF definidas correctamente.")

Funciones explícitas para TF, DF, IDF y TF-IDF definidas correctamente.


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [3]:
# 1. Construir un vocabulario con un orden definido (alfabético)
vocabulario = sorted(list(vocabulario_set))
term_to_idx = {t: i for i, t in enumerate(vocabulario)}
N_docs = len(docs_ids)

# 2. Contar las frecuencias TF
matriz_tf = calcular_tf(corpus_tokens, vocabulario, docs_ids)

# 3. Calcular DF contando cada documento una sola vez por término
vector_df = calcular_df(matriz_tf)

# 4. Calcular IDF con la función de la sección anterior
vector_idf = calcular_idf(vector_df, N_docs)

# 5. Obtener la matriz documento-término TF-IDF
matriz_tfidf_explicita = calcular_tfidf(matriz_tf, vector_idf)

print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf_explicita.shape} (Documentos x Vocabulario)")
print(f"Memoria ocupada por la matriz explícita: {matriz_tfidf_explicita.nbytes / (1024**2):.2f} MB")

# Mostrar 6 términos representativos y sus pesos TF-IDF en el primer documento
terminos_muestra = ['the', 'and', 'time', 'world', 'king', 'love']
datos_muestra = []
for t in terminos_muestra:
    if t in term_to_idx and N_docs > 0:
        idx = term_to_idx[t]
        datos_muestra.append({
            'Término': t,
            'DF(t)': int(vector_df[idx]),
            'IDF(t)': round(vector_idf[idx], 6),
            f'TF en doc[0] ({docs_ids[0]})': int(matriz_tf[0, idx]),
            'Peso TF-IDF en doc[0]': round(matriz_tfidf_explicita[0, idx], 6)
        })

display(pd.DataFrame(datos_muestra))

Dimensiones de la matriz TF-IDF: (300, 277088) (Documentos x Vocabulario)
Memoria ocupada por la matriz explícita: 634.20 MB


,Término,DF(t),IDF(t),TF en doc[0] (cache/epub/10000/pg10000.txt),Peso TF-IDF en doc[0]
0,the,300,1.000000,1144,1144.000000
1,and,300,1.000000,672,672.000000
2,time,277,1.079765,20,21.595299
3,world,300,1.000000,2,2.000000
4,king,230,1.265703,21,26.579766
5,love,261,1.139262,0,0.000000


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [4]:
# Pasamos directamente las listas de tokens ya preprocesadas mediante funciones identidad
lista_tokens_corpus = [corpus_tokens[doc_id] for doc_id in docs_ids]

vectorizador = TfidfVectorizer(
    tokenizer=lambda x: x,
    preprocessor=lambda x: x,
    token_pattern=None,
    lowercase=False,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
    vocabulary=vocabulario
)

matriz_tfidf_sklearn = vectorizador.fit_transform(lista_tokens_corpus).toarray()

# Verificación de igualdad de orden de columnas y diferencia absoluta máxima
vocab_sklearn = list(vectorizador.get_feature_names_out())
max_diff_idf = np.max(np.abs(vector_idf - vectorizador.idf_))
max_diff_matriz = np.max(np.abs(matriz_tfidf_explicita - matriz_tfidf_sklearn))

print(f"¿El orden de columnas del vocabulario coincide exactamente?: {vocabulario == vocab_sklearn}")
print(f"Máxima diferencia absoluta en vectores IDF: {max_diff_idf:.2e}")
print(f"Máxima diferencia absoluta entre ambas matrices TF-IDF: {max_diff_matriz:.2e}")

¿El orden de columnas del vocabulario coincide exactamente?: True
Máxima diferencia absoluta en vectores IDF: 0.00e+00
Máxima diferencia absoluta entre ambas matrices TF-IDF: 0.00e+00


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [5]:
consultas_info = [
    {
        "id": "Q1",
        "consulta": "ship sea captain voyage island",
        "necesidad": "Encontrar relatos de navegación marítima, capitanes de barco, travesías y exploraciones en islas."
    },
    {
        "id": "Q2",
        "consulta": "king queen war army battle soldier",
        "necesidad": "Recuperar crónicas históricas o literarias sobre monarquías, conflictos bélicos, ejércitos y batallas."
    },
    {
        "id": "Q3",
        "consulta": "love marriage family heart father mother",
        "necesidad": "Localizar novelas románticas o dramas familiares centrados en el matrimonio y relaciones entre padres e hijos."
    },
    {
        "id": "Q4",
        "consulta": "god heaven soul spirit prayer faith",
        "necesidad": "Buscar textos teológicos, religiosos o filosóficos que traten sobre la fe, el alma, la oración y lo divino."
    }
]

def vectorizar_consulta_explicita(texto_consulta, term_to_idx, idf_vector):
    tokens_q = preprocesar(texto_consulta)
    conteos_q = Counter(tokens_q)
    tf_q = np.zeros(len(term_to_idx), dtype=np.float64)
    for t, c in conteos_q.items():
        if t in term_to_idx:
            tf_q[term_to_idx[t]] = c
    return tf_q * idf_vector, tokens_q

def similitud_coseno_explicita(vec_q, matriz_docs):
    """
    Calcula la similitud coseno mediante el producto punto y las magnitudes (normas L2) de los vectores.
    """
    norma_q = np.linalg.norm(vec_q)
    normas_docs = np.linalg.norm(matriz_docs, axis=1)
    productos_punto = np.dot(matriz_docs, vec_q)
    
    denominador = norma_q * normas_docs
    similitudes = np.divide(productos_punto, denominador, out=np.zeros_like(productos_punto), where=denominador!=0)
    return similitudes, productos_punto, norma_q, normas_docs

vectores_q_explicitos = []
tokens_consultas = []

for q in consultas_info:
    vec_q, tok_q = vectorizar_consulta_explicita(q["consulta"], term_to_idx, vector_idf)
    vectores_q_explicitos.append(vec_q)
    tokens_consultas.append(tok_q)
    print(f"{q['id']}: '{q['consulta']}'")
    print(f"   Necesidad de información: {q['necesidad']}")
    print(f"   Tokens procesados: {tok_q}\n")

matriz_q_explicita = np.vstack(vectores_q_explicitos)
matriz_q_sklearn = vectorizador.transform(tokens_consultas).toarray()

# Comparación de similitudes explícitas vs scikit-learn
sim_explicita_todas = np.zeros((len(consultas_info), N_docs))
for i in range(len(consultas_info)):
    sim_explicita_todas[i], _, _, _ = similitud_coseno_explicita(matriz_q_explicita[i], matriz_tfidf_explicita)

sim_sklearn_todas = cosine_similarity(matriz_q_sklearn, matriz_tfidf_sklearn)
diff_sim = np.max(np.abs(sim_explicita_todas - sim_sklearn_todas))
print(f"Máxima diferencia absoluta en similitud coseno (Explícita vs Sklearn): {diff_sim:.2e}\n")

# Mostrar tabla detallada para la combinación Q1 y su documento más relevante del corpus
q_idx_demo = 0
d_idx_demo = int(np.argmax(sim_explicita_todas[q_idx_demo]))
doc_id_demo = docs_ids[d_idx_demo]

vec_q_demo = matriz_q_explicita[q_idx_demo]
vec_d_demo = matriz_tfidf_explicita[d_idx_demo]

indices_compartidos = np.where((vec_q_demo > 0) & (vec_d_demo > 0))[0]
terminos_compartidos = [vocabulario[i] for i in indices_compartidos]
productos_parciales = [round(vec_q_demo[i] * vec_d_demo[i], 4) for i in indices_compartidos]
prod_punto_demo = np.dot(vec_q_demo, vec_d_demo)
norma_q_demo = np.linalg.norm(vec_q_demo)
norma_d_demo = np.linalg.norm(vec_d_demo)
sim_demo = prod_punto_demo / (norma_q_demo * norma_d_demo)

print(f"Detalle de similitud coseno para {consultas_info[q_idx_demo]['id']} vs '{doc_id_demo}' ({titulos[doc_id_demo]}):")
df_desglose = pd.DataFrame([{
    'Términos compartidos': ", ".join(terminos_compartidos),
    'Productos parciales': str(dict(zip(terminos_compartidos, productos_parciales))),
    'Producto punto': round(prod_punto_demo, 4),
    'Normas': f"||q||={norma_q_demo:.4f}, ||d||={norma_d_demo:.4f}",
    'Similitud': round(sim_demo, 6)
}])
display(df_desglose)

Q1: 'ship sea captain voyage island'
   Necesidad de información: Encontrar relatos de navegación marítima, capitanes de barco, travesías y exploraciones en islas.
   Tokens procesados: ['ship', 'sea', 'captain', 'voyage', 'island']

Q2: 'king queen war army battle soldier'
   Necesidad de información: Recuperar crónicas históricas o literarias sobre monarquías, conflictos bélicos, ejércitos y batallas.
   Tokens procesados: ['king', 'queen', 'war', 'army', 'battle', 'soldier']

Q3: 'love marriage family heart father mother'
   Necesidad de información: Localizar novelas románticas o dramas familiares centrados en el matrimonio y relaciones entre padres e hijos.
   Tokens procesados: ['love', 'marriage', 'family', 'heart', 'father', 'mother']

Q4: 'god heaven soul spirit prayer faith'
   Necesidad de información: Buscar textos teológicos, religiosos o filosóficos que traten sobre la fe, el alma, la oración y lo divino.
   Tokens procesados: ['god', 'heaven', 'soul', 'spirit', 'prayer',

,Términos compartidos,Productos parciales,Producto punto,Normas,Similitud
0,"captain, island, sea, ship, voyage","{'captain': 1187.1836, 'island': 346.3506, 'se...",2146.0667,"||q||=3.4304, ||d||=9596.7038",0.065189


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [6]:
def obtener_fragmento(texto_raw, tokens_consulta, ventana=70):
    """
    Extrae un fragmento del texto original alrededor de la primera aparición de algún término de la consulta.
    """
    texto_limpio = re.sub(r'\s+', ' ', texto_raw).strip()
    for token in tokens_consulta:
        match = re.search(r'\b' + re.escape(token) + r'\b', texto_limpio, re.IGNORECASE)
        if match:
            inicio = max(0, match.start() - ventana)
            fin = min(len(texto_limpio), match.end() + ventana)
            return "..." + texto_limpio[inicio:fin] + "..."
    return texto_limpio[:2 * ventana] + "..."

rankings_tfidf = {}

for i, q in enumerate(consultas_info):
    puntuaciones_exp = sim_explicita_todas[i]
    puntuaciones_skl = sim_sklearn_todas[i]
    
    # Verificar que ambas implementaciones producen las mismas puntuaciones dentro de la tolerancia
    assert np.allclose(puntuaciones_exp, puntuaciones_skl, atol=1e-12), "Diferencia detectada entre implementaciones"
    
    # Ordenar de mayor a menor similitud
    indices_top5 = np.argsort(puntuaciones_exp)[::-1][:5]
    rankings_tfidf[q['id']] = [docs_ids[idx] for idx in indices_top5]
    
    filas_ranking = []
    for pos, doc_idx in enumerate(indices_top5, start=1):
        doc_id = docs_ids[doc_idx]
        filas_ranking.append({
            'Posición': pos,
            'Identificador': doc_id,
            'Título': titulos[doc_id],
            'Puntuación': round(puntuaciones_exp[doc_idx], 6),
            'Fragmento': obtener_fragmento(textos_originales[doc_id], tokens_consultas[i])
        })
    
    print(f"=== Top-5 Ranking TF-IDF para {q['id']}: '{q['consulta']}' ===")
    display(pd.DataFrame(filas_ranking))

=== Top-5 Ranking TF-IDF para Q1: 'ship sea captain voyage island' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10339/pg10339.txt,An Antarctic Mystery,0.065189,"...f as large as Corsica. It is safe, and easy..."
1,2,cache/epub/10008/pg10008.txt,The Mystery,0.048094,"... THE SEA RIDDLE I. DESERT SEAS II. THE ""LAU..."
2,3,cache/epub/10370/pg10370.txt,Sustained honor: The Age of Liberty Established,0.042061,... association Emigrants' wagon crossing a st...
3,4,cache/epub/1024/pg1024.txt,The Wrecker,0.029125,...n the weir. It is bold water at the mouth o...
4,5,cache/epub/10081/pg10081.txt,"The Boy Allies at Jutland; Or, The Greatest Na...",0.028059,...--heading eastward down the river. Men line...


=== Top-5 Ranking TF-IDF para Q2: 'king queen war army battle soldier' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10147/pg10147.txt,America's War for Humanity,0.039032,"...hods. (_Copyright, Committee on Public Info..."
1,2,cache/epub/1030/pg1030.txt,The Cavalier Songs and Ballads of England from...,0.035620,... overthrow. Though many thousands of these ...
2,3,cache/epub/10159/pg10159.txt,England and the War,0.033904,...so they must learn to know the English char...
3,4,cache/epub/100/pg100.txt,The Complete Works of William Shakespeare,0.033299,"... CYMBELINE THE TRAGEDY OF HAMLET, PRINCE OF..."
4,5,cache/epub/10166/pg10166.txt,"What Germany Thinks; Or, The War as Germans se...",0.031659,...utenberg.org/ebooks/10166 Credits: Produced...


=== Top-5 Ranking TF-IDF para Q3: 'love marriage family heart father mother' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10216/pg10216.txt,Poems,0.042387,"...est eyes in the world"" would brighten over ..."
1,2,cache/epub/10063/pg10063.txt,Happiness and Marriage,0.042116,... happy and peaceful. I long for a wedded li...
2,3,cache/epub/10109/pg10109.txt,The Unspeakable Gentleman,0.040287,... of animation came to change him. Politely ...
3,4,cache/epub/10376/pg10376.txt,American Indian Stories,0.037789,"...pt in secret, wishing I had gone West, to b..."
4,5,cache/epub/1019/pg1019.txt,"Poems by Currer, Ellis, and Acton Bell",0.036105,"...t, urge-- Rome's servile slave, and Judah's..."


=== Top-5 Ranking TF-IDF para Q4: 'god heaven soul spirit prayer faith' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,cache/epub/10325/pg10325.txt,The Gospel of the Pentateuch: A Set of Parish ...,0.082342,...Scripture asserts that those who wrote it w...
1,2,cache/epub/10116/pg10116.txt,All Saints' Day and Other Sermons,0.076668,...in their hands; and cried with a loud voice...
2,3,cache/epub/10326/pg10326.txt,David: Five Sermons,0.054713,...on of his native land. He who followed the ...
3,4,cache/epub/10314/pg10314.txt,The Sonnets of Michael Angelo Buonarroti and T...,0.052307,"... constructed for himself a new theology, fo..."
4,5,cache/epub/1019/pg1019.txt,"Poems by Currer, Ellis, and Acton Bell",0.049120,...hrist?-to me unknown His lineage--doctrine-...


### 7. Implementación de BM25 y Comparación de Rankings

Para cumplir con los objetivos de implementar **BM25**, comprender sus componentes (saturación de frecuencia de términos $k_1$ y normalización por longitud de documento $b$) y comparar los rankings frente a TF-IDF para las mismas consultas, utilizamos la formulación estándar:

$$
\operatorname{BM25}(q, d) = \sum_{t \in q} IDF_{BM25}(t) \cdot \frac{f(t, d) \cdot (k_1 + 1)}{f(t, d) + k_1 \cdot \left(1 - b + b \cdot \frac{|d|}{\text{avgdl}}\right)}
$$

donde:
- $IDF_{BM25}(t) = \ln\left(1 + \frac{N - DF(t) + 0.5}{DF(t) + 0.5}\right)$
- $|d|$ es la longitud del documento $d$ y $\text{avgdl}$ es la longitud media de los documentos en el corpus.
- Parámetros habituales: $k_1 = 1.5$ y $b = 0.75$.

In [7]:
def puntuar_bm25_consulta(tokens_q, matriz_tf, df_vector, longitudes_docs, term_to_idx, k1=1.5, b=0.75):
    """
    Calcula la puntuación BM25 de todos los documentos para una consulta específica
    operando únicamente sobre las columnas de los términos consultados (eficiente en RAM).
    """
    N = matriz_tf.shape[0]
    avgdl = np.mean(longitudes_docs)
    norm_longitud = (1.0 - b) + b * (np.array(longitudes_docs, dtype=np.float64) / avgdl)
    
    puntuaciones = np.zeros(N, dtype=np.float64)
    for t in tokens_q:
        if t in term_to_idx:
            idx = term_to_idx[t]
            df_t = df_vector[idx]
            idf_t = np.log(1.0 + (N - df_t + 0.5) / (df_t + 0.5))
            
            tf_t = matriz_tf[:, idx]
            numerador = tf_t * (k1 + 1.0)
            denominador = tf_t + k1 * norm_longitud
            
            puntuaciones += idf_t * np.divide(numerador, denominador, out=np.zeros_like(numerador), where=denominador!=0)
            
    return puntuaciones

rankings_bm25 = {}

for i, q in enumerate(consultas_info):
    tokens_q = tokens_consultas[i]
    puntuaciones_bm25 = puntuar_bm25_consulta(tokens_q, matriz_tf, vector_df, longitudes, term_to_idx, k1=1.5, b=0.75)
    
    indices_top5_bm25 = np.argsort(puntuaciones_bm25)[::-1][:5]
    rankings_bm25[q['id']] = [docs_ids[idx] for idx in indices_top5_bm25]
    
    filas_comparacion = []
    for pos in range(5):
        doc_tfidf = rankings_tfidf[q['id']][pos]
        idx_tfidf = docs_ids.index(doc_tfidf)
        
        doc_bm25 = rankings_bm25[q['id']][pos]
        idx_bm25 = docs_ids.index(doc_bm25)
        
        filas_comparacion.append({
            'Posición': pos + 1,
            'Top-5 TF-IDF (Coseno)': f"{doc_tfidf} ({sim_explicita_todas[i][idx_tfidf]:.4f})",
            'Título (TF-IDF)': titulos[doc_tfidf],
            'Top-5 BM25': f"{doc_bm25} ({puntuaciones_bm25[idx_bm25]:.4f})",
            'Título (BM25)': titulos[doc_bm25],
            '¿Coincide?': doc_tfidf == doc_bm25
        })
        
    print(f"=== Comparación de Rankings (TF-IDF vs BM25) para {q['id']}: '{q['consulta']}' ===")
    display(pd.DataFrame(filas_comparacion))

=== Comparación de Rankings (TF-IDF vs BM25) para Q1: 'ship sea captain voyage island' ===


,Posición,Top-5 TF-IDF (Coseno),Título (TF-IDF),Top-5 BM25,Título (BM25),¿Coincide?
0,1,cache/epub/10339/pg10339.txt (0.0652),An Antarctic Mystery,cache/epub/10339/pg10339.txt (6.3861),An Antarctic Mystery,True
1,2,cache/epub/10008/pg10008.txt (0.0481),The Mystery,cache/epub/10121/pg10121.txt (6.2592),The Literature of Arabia,False
2,3,cache/epub/10370/pg10370.txt (0.0421),Sustained honor: The Age of Liberty Established,cache/epub/10103/pg10103.txt (6.2447),"The Great Events by Famous Historians, Volume 08",False
3,4,cache/epub/1024/pg1024.txt (0.0291),The Wrecker,cache/epub/10387/pg10387.txt (6.2170),A Century Too Soon: The Age of Tyranny,False
4,5,cache/epub/10081/pg10081.txt (0.0281),"The Boy Allies at Jutland; Or, The Greatest Na...",cache/epub/10008/pg10008.txt (6.1874),The Mystery,False


=== Comparación de Rankings (TF-IDF vs BM25) para Q2: 'king queen war army battle soldier' ===


,Posición,Top-5 TF-IDF (Coseno),Título (TF-IDF),Top-5 BM25,Título (BM25),¿Coincide?
0,1,cache/epub/10147/pg10147.txt (0.0390),America's War for Humanity,cache/epub/10351/pg10351.txt (6.1594),"The Works of Samuel Johnson, LL.D. Volume 10",False
1,2,cache/epub/1030/pg1030.txt (0.0356),The Cavalier Songs and Ballads of England from...,cache/epub/10114/pg10114.txt (6.1476),"The Great Events by Famous Historians, Volume 02",False
2,3,cache/epub/10159/pg10159.txt (0.0339),England and the War,cache/epub/10024/pg10024.txt (6.1199),Beneath the Banner: Being Narratives of Noble ...,False
3,4,cache/epub/100/pg100.txt (0.0333),The Complete Works of William Shakespeare,cache/epub/100/pg100.txt (6.0679),The Complete Works of William Shakespeare,True
4,5,cache/epub/10166/pg10166.txt (0.0317),"What Germany Thinks; Or, The War as Germans se...",cache/epub/10147/pg10147.txt (6.0675),America's War for Humanity,False


=== Comparación de Rankings (TF-IDF vs BM25) para Q3: 'love marriage family heart father mother' ===


,Posición,Top-5 TF-IDF (Coseno),Título (TF-IDF),Top-5 BM25,Título (BM25),¿Coincide?
0,1,cache/epub/10216/pg10216.txt (0.0424),Poems,cache/epub/10063/pg10063.txt (3.1762),Happiness and Marriage,False
1,2,cache/epub/10063/pg10063.txt (0.0421),Happiness and Marriage,cache/epub/10365/pg10365.txt (3.1701),Precaution: A Novel,False
2,3,cache/epub/10109/pg10109.txt (0.0403),The Unspeakable Gentleman,cache/epub/10130/pg10130.txt (3.1425),The Works of Charles and Mary Lamb — Volume 3,False
3,4,cache/epub/10376/pg10376.txt (0.0378),American Indian Stories,cache/epub/10121/pg10121.txt (3.1284),The Literature of Arabia,False
4,5,cache/epub/1019/pg1019.txt (0.0361),"Poems by Currer, Ellis, and Acton Bell",cache/epub/10129/pg10129.txt (3.1252),Excellent Women,False


=== Comparación de Rankings (TF-IDF vs BM25) para Q4: 'god heaven soul spirit prayer faith' ===


,Posición,Top-5 TF-IDF (Coseno),Título (TF-IDF),Top-5 BM25,Título (BM25),¿Coincide?
0,1,cache/epub/10325/pg10325.txt (0.0823),The Gospel of the Pentateuch: A Set of Parish ...,cache/epub/1019/pg1019.txt (4.7365),"Poems by Currer, Ellis, and Acton Bell",False
1,2,cache/epub/10116/pg10116.txt (0.0767),All Saints' Day and Other Sermons,cache/epub/10058/pg10058.txt (4.7236),The Divine Office: A Study of the Roman Breviary,False
2,3,cache/epub/10326/pg10326.txt (0.0547),David: Five Sermons,cache/epub/10129/pg10129.txt (4.7225),Excellent Women,False
3,4,cache/epub/10314/pg10314.txt (0.0523),The Sonnets of Michael Angelo Buonarroti and T...,cache/epub/10375/pg10375.txt (4.7223),England's Antiphon,False
4,5,cache/epub/1019/pg1019.txt (0.0491),"Poems by Currer, Ellis, and Acton Bell",cache/epub/10314/pg10314.txt (4.7131),The Sonnets of Michael Angelo Buonarroti and T...,False
